In [ ]:
# 引入依赖库以及做库的基础配置

# 数据分析库
# import numpy as np

import pandas as pd

# import matplotlib.pyplot as plt

# plt.rcParams["font.family"] = "Heiti TC"  # matplotlib 中文支持 on mac

# 数据可视化库
import plotly.io as pio
import plotly.express as px

pio.renderers.default = "notebook"


# 自定义库
from utils import get_data_file_path, load_ctg
from entry import EntryList

# 其他库
from rich import print
import datetime
from collections import defaultdict

In [ ]:
# 全局常量
EXPENSE_TYPE = "支出"

In [ ]:
# 读入配置
ctg_conf = load_ctg()
print(ctg_conf)

In [ ]:
# 读入数据
data_file_path = get_data_file_path()
data = EntryList.from_csv_file(data_file_path)

In [ ]:
# 数据过滤

# 1. 方案一：保持全部数据——不处理
print("adhoc")

# 2. 方案二：查看上一年年终总结
# last_year = datetime.datetime.now().year - 1
# print(f"上一年为: {last_year}年")
# data = data.filter(lambda item: item.date.year == last_year)

# 3. 方案三：按照tag过滤
# data = data.filter(lambda item: "乌兰布统" in item.tags)

# 4. 方案四: 按照时间范围过滤
# start_date = datetime.datetime(2026, 1, 1)
# end_date = datetime.datetime(2026, 6, 6)
# data = data.filter(lambda item: start_date <= item.date <= end_date)

# 5. 方案五: 过去一年的数据
# today = datetime.datetime.now()
# one_year_ago = today - datetime.timedelta(days=365)
# data = data.filter(lambda item: one_year_ago <= item.date <= today)

# ==============================================================================

expense_data = data.filter(lambda item: item.type == EXPENSE_TYPE)

In [ ]:
# 输出基本数据
print(f"收入总额: {data.filter(lambda item: item.type == "收入").sum():,.2f}元")
print(f"消费总额: {-1 * data.filter(lambda item: item.type == "支出").sum():,.2f}元")
print(f"结余总额: {data.sum():,.2f}元")

In [ ]:
# 消费分布饼图
# ref: https://plotly.com/python/sunburst-charts/

rows = []
for entry in expense_data:
    ctg_0 = entry.categorys[0] if len(entry.categorys) > 0 else ""
    ctg_1 = entry.categorys[1] if len(entry.categorys) > 1 else ""
    if ctg_1 in ("早饭", "午饭", "晚饭"):
        ctg_1 = "正餐"
    rows.append(
        {
            "ctg_0": ctg_0,
            "ctg_1": ctg_1,
            "amount": entry.amount,
        }
    )

df = pd.DataFrame(rows)
df = df.groupby(["ctg_0", "ctg_1"], as_index=False, dropna=False)["amount"].sum()
df["type"] = EXPENSE_TYPE


fig = px.sunburst(
    df,
    path=[
        "type",
        "ctg_0",
        "ctg_1",
    ],
    values="amount",
)

fig.update_traces(textinfo="label+percent entry")  # 显示中添加百分比

fig.write_html("output/expense_sunburst.html")

fig.show()

In [ ]:
# 标签分布图

expense_by_tag = defaultdict(float)
for entry in expense_data:
    # tags = entry.tags if entry.tags else [""]
    tags = entry.tags
    for tag in tags:
        expense_by_tag[tag] += entry.amount

df = pd.DataFrame.from_dict(dict(expense_by_tag), orient="index", columns=["amount"])
df = df.reset_index().rename(columns={"index": "tag"})

fig = px.bar(
    df,
    x="tag",
    y="amount",
    text_auto=".2s",
    title="标签分布",
)
fig.update_traces(textfont_size=12, textangle=0, textposition="outside", cliponaxis=False)
# fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
fig.write_html("output/expense_bar.html")
fig.show()